# Time-window feasibility of the benchmark instances

Checks every instance in `data/raw/solomon/` and decides which to keep.

**The keep rule: service must be completed inside the customer's window**, i.e.
`ready_time + service_time <= due_date`. An instance is **kept** only if every customer passes.

This is stricter than the benchmarks' own convention, where the due date is the latest *start* of service,
so service may run past it. The published solutions and the solver's lateness rule use that convention.
The stricter rule reflects, for example, a store that closes at its due time.

Supporting checks, using Solomon travel time (1 distance unit = 1 minute, leaving the depot at its ready time):

| check | condition |
|---|---|
| `fits_window` | `ready + service <= due` (**the keep/discard rule**) |
| `reachable` | `max(depot_ready + t(0,i), ready) + service <= due` |
| `can_return` | `max(depot_ready + t(0,i), ready) + service + t(i,0) <= depot_due` |
| `demand_ok` | `demand <= vehicle capacity` |

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

# repository root: the first parent folder holding data/raw
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "processed" / "time_window_feasibility.csv"
COLS = ["cust", "x", "y", "demand", "ready", "due", "service"]
print(ROOT)

C:\Users\user\ML\Optimisation\EV_VRPTW\EV_VRPTW


## Load the instances

In [2]:
# vehicle capacity per Solomon family (the CSVs carry no vehicle section)
SOLOMON_CAPACITY = {"C1": 200, "C2": 700, "R1": 200, "R2": 1000, "RC1": 200, "RC2": 1000}

def read_solomon_csv(path):
    d = pd.read_csv(path).iloc[:, :7].dropna()   # some files carry extra trailing columns
    d.columns = COLS
    return d.reset_index(drop=True)

instances = []
for path in sorted((RAW / "solomon").glob("*.csv")):
    family = re.match(r"([a-z]+\d)", path.stem, re.I).group(1).upper()
    instances.append(dict(instance=path.stem, family=family, data=read_solomon_csv(path),
                          capacity=SOLOMON_CAPACITY[family]))
print(len(instances), "instances:", ", ".join(i["instance"] for i in instances))

16 instances: c107, c108, c109, r101, r104, r105, r204, r205, r208, rc101, rc104, rc105, rc107, rc201, rc204, rc207


## Check every customer

In [3]:
def check(inst):
    d = inst["data"]
    depot, cust = d.iloc[0], d.iloc[1:].copy()
    t = np.hypot(cust.x - depot.x, cust.y - depot.y)          # Solomon: 1 unit = 1 minute
    start = np.maximum(depot.ready + t, cust.ready)              # earliest service start
    cust["window"] = cust.due - cust.ready
    cust["fits_window"] = cust.ready + cust.service <= cust.due
    cust["reachable"] = start + cust.service <= cust.due
    cust["can_return"] = start + cust.service + t <= depot.due
    cust["demand_ok"] = cust.demand <= inst["capacity"]
    return cust

customers = pd.concat([check(i).assign(family=i["family"], instance=i["instance"]) for i in instances],
                      ignore_index=True)

CHECKS = ["fits_window", "reachable", "can_return", "demand_ok"]
summary = (customers.groupby(["family", "instance"], sort=False)
           .agg(customers=("cust", "size"),
                narrow_windows=("fits_window", lambda s: int((~s).sum())),
                shortest_window_min=("window", "min"),
                **{f"all_{c}": (c, "all") for c in CHECKS})
           .reset_index())
summary["keep"] = summary.all_fits_window
summary

,family,instance,customers,narrow_windows,shortest_window_min,all_fits_window,all_reachable,all_can_return,all_demand_ok,keep
0,C1,c107,100,0,180.0,True,True,True,True,True
1,C1,c108,100,0,149.0,True,True,True,True,True
2,C1,c109,100,0,360.0,True,True,True,True,True
3,R1,r101,100,0,73.0,True,True,True,True,True
4,R1,r104,100,0,10.0,True,False,True,True,True
5,R1,r105,100,0,30.0,True,True,True,True,True
6,R2,r204,100,0,27.0,True,True,True,True,True
7,R2,r205,100,0,240.0,True,True,True,True,True
8,R2,r208,100,0,240.0,True,True,True,True,True
9,RC1,rc101,100,0,30.0,True,True,True,True,True


`narrow_windows`: customers whose window is shorter than their service time.

All 16 should be kept: no customer's service is longer than its window.

In [4]:
print(f"keep {summary.keep.sum()} of {len(summary)}")
if (~summary.keep).any():
    print("DISCARD:", ", ".join(summary[~summary.keep].instance))

keep 16 of 16


## Other checks

Instances that pass the keep rule but fail a supporting check. These are **kept**. The failures are
`reachable` only, and only by fractions of a minute at Solomon's travel speed (1 unit per minute, about 3 km/h).
For example, r104 customer 93 (window 18-28, service 10) is 18.4 units from the depot, so even served first it
finishes at 28.4. In our model the vehicle covers those 920 m in about 2 minutes, so every customer is
reachable.

In [5]:
other = summary[summary.keep & ~summary[[f"all_{c}" for c in CHECKS[1:]]].all(axis=1)]
late = customers[customers.instance.isin(other.instance) & ~customers.reachable]
other if len(other) else print("none")

,family,instance,customers,narrow_windows,shortest_window_min,all_fits_window,all_reachable,all_can_return,all_demand_ok,keep
4,R1,r104,100,0,10.0,True,False,True,True,True


In [6]:
late[["instance", "cust", "x", "y", "ready", "due", "service"]] if len(late) else None

,instance,cust,x,y,ready,due,service
491,r104,93.0,22.0,22.0,18.0,28.0,10.0


## Save

In [7]:
summary.to_csv(OUT, index=False)
print(f"wrote {OUT.relative_to(ROOT)} ({len(summary)} instances, {summary.keep.sum()} kept)")

wrote data\processed\time_window_feasibility.csv (16 instances, 16 kept)
